# Model AI Meal Plan — Functional API + TensorBoard

Notebook ini berisi alur final pembuatan model AI untuk klasifikasi **Recommended Meal Plan** dengan TensorFlow Functional API.

Notebook ini juga menambahkan side quest **TensorBoard Integration**, yaitu menyimpan log training dan validation agar proses pelatihan dapat dipantau melalui dashboard TensorBoard.

Output model:
- `High-Protein Diet`
- `Low-Carb Diet`
- `Low-Fat Diet`

Catatan: versi ini **tidak menggunakan `tf.GradientTape`** agar alur training tetap mudah dipahami oleh tim. Training dilakukan menggunakan `model.fit()` dengan callback TensorBoard.

## 1. Mount Google Drive

Jalankan cell ini jika notebook digunakan di Google Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Import Library

In [ ]:
# =====================================================
# IMPORT LIBRARY
# =====================================================

import os
import json
import random
import datetime
import zipfile

import numpy as np
import pandas as pd

import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

## 3. Konfigurasi Path dan Random Seed

Sesuaikan path dataset jika lokasi file berbeda.

In [ ]:
# =====================================================
# CONFIGURATION
# =====================================================

CLEAN_DATA_PATH = '/content/drive/MyDrive/new_df_clean.csv'
RAW_DATA_PATH = '/content/drive/MyDrive/new_df_eda.csv'

ARTIFACT_DIR = 'meal_plan_artifacts_tensorboard'
os.makedirs(ARTIFACT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('Artifact directory:', ARTIFACT_DIR)

## 4. Load Dataset

- `df_clean`: dataset yang sudah melalui preprocessing/normalisasi, digunakan untuk training.
- `df_raw`: dataset asli, digunakan untuk mengambil nilai minimum dan maksimum fitur numerik saat inference.

In [ ]:
df_clean = pd.read_csv(CLEAN_DATA_PATH)
df_raw = pd.read_csv(RAW_DATA_PATH)

print("df_clean shape:", df_clean.shape)
print("df_raw shape:", df_raw.shape)

df_clean.head()

## 5. Cek Informasi Dataset

In [ ]:
df_clean.info()

print("\nJumlah data:", df_clean.shape[0])
print("Jumlah kolom:", df_clean.shape[1])

## 6. Menentukan Target Meal Plan

Target awal berbentuk **one-hot encoding**, yaitu 3 kolom meal plan.  
Setiap baris hanya memiliki satu label aktif bernilai `1`.

In [ ]:
meal_plan_cols = [
    'Recommended_Meal_Plan_High-Protein Diet',
    'Recommended_Meal_Plan_Low-Carb Diet',
    'Recommended_Meal_Plan_Low-Fat Diet'
]

print("Distribusi target:")
print(df_clean[meal_plan_cols].sum())

## 7. Mapping Target

Target one-hot akan diubah menjadi label angka:
- `0` = High-Protein Diet
- `1` = Low-Carb Diet
- `2` = Low-Fat Diet

In [ ]:
meal_plan_mapping = {
    'Recommended_Meal_Plan_High-Protein Diet': 0,
    'Recommended_Meal_Plan_Low-Carb Diet': 1,
    'Recommended_Meal_Plan_Low-Fat Diet': 2
}

reverse_meal_plan_mapping = {
    0: 'High-Protein Diet',
    1: 'Low-Carb Diet',
    2: 'Low-Fat Diet'
}

## 8. Encode Target

In [ ]:
y_label = df_clean[meal_plan_cols].idxmax(axis=1)
y = y_label.map(meal_plan_mapping)

print("Cek target encoded:")
print(y.value_counts().sort_index())

print("\nMapping:")
for k, v in reverse_meal_plan_mapping.items():
    print(k, "=", v)

## 9. Menentukan Fitur X dan Menghapus Data Leakage

Kolom yang dihapus:
- Kolom target meal plan.
- Kolom rekomendasi nutrisi (`Recommended_*`) karena itu adalah hasil rekomendasi, bukan input user.
- Kolom BMI dan turunannya jika sistem final tidak meminta input BMI.

In [ ]:
drop_columns = [
    'Recommended_Meal_Plan_High-Protein Diet',
    'Recommended_Meal_Plan_Low-Carb Diet',
    'Recommended_Meal_Plan_Low-Fat Diet',

    'Recommended_Calories',
    'Recommended_Protein',
    'Recommended_Carbs',
    'Recommended_Fats',

    'BMI',
    'BMI_Status',
    'BMI_Category_Index',
    'Weight_Height_Ratio'
]

X = df_clean.drop(columns=drop_columns)

print("Jumlah fitur:", X.shape[1])
print("Shape X:", X.shape)
print("Shape y:", y.shape)

print("\nFitur yang digunakan:")
print(list(X.columns))

## 10. Train-Test Split

Menggunakan `stratify=y` agar distribusi target pada data train dan test tetap seimbang.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Shape Data")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

## 11. Membuat Model dengan TensorFlow Functional API

Model ini menggunakan arsitektur MLP untuk data tabular.

Arsitektur:
- Input layer sesuai jumlah fitur.
- Dense 128 + Dropout.
- Dense 64 + Dropout.
- Dense 32.
- Output Dense 3 dengan aktivasi `softmax`.

In [ ]:
num_features = X_train.shape[1]
num_classes = len(np.unique(y_train))

inputs = tf.keras.Input(shape=(num_features,), name="input_features")

x = tf.keras.layers.Dense(128, activation="relu", name="dense_128")(inputs)
x = tf.keras.layers.Dropout(0.3, name="dropout_1")(x)

x = tf.keras.layers.Dense(64, activation="relu", name="dense_64")(x)
x = tf.keras.layers.Dropout(0.3, name="dropout_2")(x)

x = tf.keras.layers.Dense(32, activation="relu", name="dense_32")(x)

outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="meal_plan_output")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name="functional_meal_plan_mlp")

model.summary()

## 12. Compile Model

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 13. Callback Training dan TensorBoard Logging

Pada bagian ini digunakan beberapa callback:

- `EarlyStopping`: menghentikan training jika `val_accuracy` tidak membaik.
- `ModelCheckpoint`: menyimpan model terbaik berdasarkan `val_accuracy`.
- `TensorBoard`: menyimpan log training untuk divisualisasikan pada dashboard TensorBoard.

Side quest TensorBoard dipenuhi pada bagian ini karena metrik `loss`, `accuracy`, `val_loss`, dan `val_accuracy` dicatat otomatis selama training.

In [ ]:
# =====================================================
# CALLBACK TRAINING + TENSORBOARD
# =====================================================

log_dir = os.path.join(
    ARTIFACT_DIR,
    'logs',
    'fit',
    datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(ARTIFACT_DIR, 'best_meal_plan_model.keras'),
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.TensorBoard(
        log_dir=log_dir,
        histogram_freq=1,
        write_graph=True,
        update_freq='epoch'
    )
]

print('TensorBoard log directory:', log_dir)

## 14. Training Model

Training dilakukan menggunakan `model.fit()` karena versi backup ini fokus pada side quest TensorBoard, bukan GradientTape.

Selama training, TensorBoard callback akan menyimpan log metrik ke folder `logs/fit`.

In [ ]:
history = model.fit(
    X_train.astype("float32"),
    y_train.astype("int32"),
    validation_data=(X_test.astype("float32"), y_test.astype("int32")),
    epochs=70,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

## 15. Visualisasi Training Curve

In [ ]:
def plot_training_curves(history):
    history_dict = history.history
    epochs = range(1, len(history_dict["loss"]) + 1)

    plt.figure(figsize=(7, 5))
    plt.plot(epochs, history_dict["loss"], label="Training Loss")
    plt.plot(epochs, history_dict["val_loss"], label="Validation Loss")
    plt.title("Training and Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure(figsize=(7, 5))
    plt.plot(epochs, history_dict["accuracy"], label="Training Accuracy")
    plt.plot(epochs, history_dict["val_accuracy"], label="Validation Accuracy")
    plt.title("Training and Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_training_curves(history)

## 16. Evaluasi Model

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test.astype("float32"),
    y_test.astype("int32"),
    verbose=1
)

print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")

## 17. Classification Report

In [ ]:
y_pred_prob = model.predict(X_test.astype("float32"))
y_pred = np.argmax(y_pred_prob, axis=1)

target_names = [
    "High-Protein Diet",
    "Low-Carb Diet",
    "Low-Fat Diet"
]

print(classification_report(
    y_test,
    y_pred,
    target_names=target_names
))

## 18. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=target_names
)

fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, values_format="d")
plt.title("Confusion Matrix")
plt.show()

## 19. Simpan Model dan Artifact Pendukung

File yang disimpan:
- `meal_plan_model.keras`: model TensorFlow final.
- `feature_columns.json`: urutan fitur yang harus digunakan saat inference.
- `meal_plan_mapping.json`: mapping label output.
- `scaler_info.json`: nilai min dan max dari dataset raw untuk normalisasi input user.
- `model_metadata.json`: ringkasan informasi model.

In [ ]:
model_path = os.path.join(ARTIFACT_DIR, "meal_plan_model.keras")
feature_columns_path = os.path.join(ARTIFACT_DIR, "feature_columns.json")
mapping_path = os.path.join(ARTIFACT_DIR, "meal_plan_mapping.json")
scaler_path = os.path.join(ARTIFACT_DIR, "scaler_info.json")
metadata_path = os.path.join(ARTIFACT_DIR, "model_metadata.json")

model.save(model_path)

with open(feature_columns_path, "w") as f:
    json.dump(list(X.columns), f, indent=2)

with open(mapping_path, "w") as f:
    json.dump(reverse_meal_plan_mapping, f, indent=2)

numeric_cols = [
    "Age",
    "Height_cm",
    "Weight_kg",
    "Blood_Pressure_Systolic",
    "Blood_Pressure_Diastolic",
    "Cholesterol_Level",
    "Blood_Sugar_Level",
    "Daily_Steps",
    "Exercise_Frequency",
    "Sleep_Hours",
    "Caloric_Intake",
    "Protein_Intake",
    "Carbohydrate_Intake",
    "Fat_Intake"
]

scaler_info = {}

for col in numeric_cols:
    scaler_info[col] = {
        "min": float(df_raw[col].min()),
        "max": float(df_raw[col].max())
    }

with open(scaler_path, "w") as f:
    json.dump(scaler_info, f, indent=2)

model_metadata = {
    "model_name": "functional_meal_plan_mlp",
    "framework": "TensorFlow/Keras",
    "api_style": "Functional API",
    "task": "Multiclass classification",
    "target_classes": reverse_meal_plan_mapping,
    "num_features": int(X.shape[1]),
    "drop_columns": drop_columns,
    "numeric_columns": numeric_cols,
    "test_accuracy": float(test_accuracy),
    "test_loss": float(test_loss)
}

with open(metadata_path, "w") as f:
    json.dump(model_metadata, f, indent=2)

print("Model dan artifact berhasil disimpan ke:")
print(ARTIFACT_DIR)

## 20. Inference Function

Bagian ini digunakan untuk mencoba prediksi dengan data baru.

Catatan penting:
- Input user masih berupa angka asli, sehingga harus dinormalisasi menggunakan `scaler_info`.
- Fitur kategorikal diubah ke one-hot encoding.
- Urutan kolom harus sama persis dengan `feature_columns`.

In [ ]:
def minmax_scale(value, min_val, max_val, clip=True):
    if max_val == min_val:
        return 0.0

    scaled = (value - min_val) / (max_val - min_val)

    if clip:
        scaled = max(0.0, min(1.0, scaled))

    return scaled


def scale_feature(column_name, value, scaler_info):
    min_val = scaler_info[column_name]["min"]
    max_val = scaler_info[column_name]["max"]
    return minmax_scale(value, min_val, max_val, clip=True)


def build_sample(input_data, feature_columns, scaler_info):
    sample = {col: 0 for col in feature_columns}

    # Numeric features
    numeric_mapping = {
        "Age": "age",
        "Height_cm": "height",
        "Weight_kg": "weight",
        "Blood_Pressure_Systolic": "bp_sys",
        "Blood_Pressure_Diastolic": "bp_dia",
        "Cholesterol_Level": "cholesterol",
        "Blood_Sugar_Level": "blood_sugar",
        "Daily_Steps": "daily_steps",
        "Exercise_Frequency": "exercise_frequency",
        "Sleep_Hours": "sleep_hours",
        "Caloric_Intake": "caloric_intake",
        "Protein_Intake": "protein_intake",
        "Carbohydrate_Intake": "carb_intake",
        "Fat_Intake": "fat_intake"
    }

    for model_col, input_key in numeric_mapping.items():
        sample[model_col] = scale_feature(model_col, input_data[input_key], scaler_info)

    # Gender
    if input_data["gender"] == "Male":
        sample["Gender_Male"] = 1
    elif input_data["gender"] == "Other":
        sample["Gender_Other"] = 1
    # Female = baseline, jadi Gender_Male dan Gender_Other tetap 0

    # Chronic disease
    chronic_map = {
        "Heart Disease": "Chronic_Disease_Heart Disease",
        "Hypertension": "Chronic_Disease_Hypertension",
        "No Chronic Disease": "Chronic_Disease_No Chronic Disease",
        "Obesity": "Chronic_Disease_Obesity"
    }
    chronic_col = chronic_map.get(input_data["chronic_disease"])
    if chronic_col:
        sample[chronic_col] = 1

    # Genetic risk
    if input_data["genetic_risk"] == "Yes":
        sample["Genetic_Risk_Factor_Yes"] = 1

    # Allergies
    allergies_map = {
        "Lactose Intolerance": "Allergies_Lactose Intolerance",
        "No Allergies": "Allergies_No Allergies",
        "Nut Allergy": "Allergies_Nut Allergy"
    }
    allergy_col = allergies_map.get(input_data["allergies"])
    if allergy_col:
        sample[allergy_col] = 1

    # Alcohol and smoking
    if input_data["alcohol"] == "Yes":
        sample["Alcohol_Consumption_Yes"] = 1

    if input_data["smoking"] == "Yes":
        sample["Smoking_Habit_Yes"] = 1

    # Dietary habit
    dietary_map = {
        "Regular": "Dietary_Habits_Regular",
        "Vegan": "Dietary_Habits_Vegan",
        "Vegetarian": "Dietary_Habits_Vegetarian"
    }
    dietary_col = dietary_map.get(input_data["dietary_habit"])
    if dietary_col:
        sample[dietary_col] = 1

    # Preferred cuisine
    cuisine_map = {
        "Indian": "Preferred_Cuisine_Indian",
        "Mediterranean": "Preferred_Cuisine_Mediterranean",
        "Western": "Preferred_Cuisine_Western"
    }
    cuisine_col = cuisine_map.get(input_data["preferred_cuisine"])
    if cuisine_col:
        sample[cuisine_col] = 1

    # Food aversion
    aversion_map = {
        "Salty": "Food_Aversions_Salty",
        "Spicy": "Food_Aversions_Spicy",
        "Sweet": "Food_Aversions_Sweet"
    }
    aversion_col = aversion_map.get(input_data["food_aversion"])
    if aversion_col:
        sample[aversion_col] = 1

    sample_df = pd.DataFrame([sample])
    sample_df = sample_df.reindex(columns=feature_columns, fill_value=0)

    return sample_df.astype("float32")


def predict_meal_plan(input_data, model, feature_columns, scaler_info, reverse_mapping):
    sample_df = build_sample(input_data, feature_columns, scaler_info)

    prediction_prob = model.predict(sample_df, verbose=0)[0]
    predicted_class = int(np.argmax(prediction_prob))
    predicted_meal_plan = reverse_mapping[predicted_class]

    probabilities = {
        reverse_mapping[i]: float(prediction_prob[i])
        for i in range(len(prediction_prob))
    }

    sorted_probs = np.sort(prediction_prob)[::-1]
    top_prob = float(sorted_probs[0])
    second_prob = float(sorted_probs[1])

    note = "Prediction confidence is good."
    if top_prob < 0.60 or (top_prob - second_prob) < 0.10:
        note = "Prediction confidence is low or close to another class."

    return {
        "recommended_meal_plan": predicted_meal_plan,
        "confidence": top_prob,
        "probabilities": probabilities,
        "note": note,
        "sample_data": sample_df
    }

## 21. Contoh Inference Data Baru

In [ ]:
new_user_data = {
    "age": 23,
    "height": 157,
    "weight": 46,

    "bp_sys": 120,
    "bp_dia": 80,

    "cholesterol": 200,
    "blood_sugar": 100,

    "daily_steps": 9000,
    "exercise_frequency": 6,
    "sleep_hours": 7,

    "caloric_intake": 2500,
    "protein_intake": 180,
    "carb_intake": 180,
    "fat_intake": 50,

    "gender": "Female",
    "chronic_disease": "No Chronic Disease",
    "genetic_risk": "No",
    "allergies": "No Allergies",
    "alcohol": "No",
    "smoking": "No",
    "dietary_habit": "Regular",
    "preferred_cuisine": "Western",
    "food_aversion": "Sweet"
}

result = predict_meal_plan(
    input_data=new_user_data,
    model=model,
    feature_columns=list(X.columns),
    scaler_info=scaler_info,
    reverse_mapping=reverse_meal_plan_mapping
)

print("==========================")
print("HASIL PREDIKSI")
print("==========================")
print("Recommended Meal Plan :", result["recommended_meal_plan"])
print(f"Confidence            : {result['confidence']:.4f}")
print("Note                  :", result["note"])

print("\nProbabilitas tiap kelas:")
for label, prob in result["probabilities"].items():
    print(f"{label}: {prob:.4f}")

## 22. Menjalankan TensorBoard di Colab

Jalankan cell berikut setelah training selesai untuk membuka dashboard TensorBoard.

Dashboard ini dapat digunakan untuk memantau:
- training loss
- training accuracy
- validation loss
- validation accuracy
- graph model
- histogram bobot model

In [ ]:
%load_ext tensorboard
%tensorboard --logdir meal_plan_artifacts_tensorboard/logs/fit

## 23. Menyimpan Log TensorBoard untuk Repository

Side quest meminta log TensorBoard disertakan dalam repository akhir. Cell berikut akan membuat file ZIP dari folder log TensorBoard.

File `tensorboard_logs.zip` bisa dimasukkan ke GitHub/repository sebagai bukti bahwa training log sudah dibuat.

In [ ]:
# =====================================================
# ZIP TENSORBOARD LOGS
# =====================================================

logs_dir = os.path.join(ARTIFACT_DIR, 'logs')
zip_path = os.path.join(ARTIFACT_DIR, 'tensorboard_logs.zip')

if os.path.exists(logs_dir):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(logs_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, ARTIFACT_DIR)
                zipf.write(file_path, arcname)

    print('TensorBoard logs saved to:', zip_path)
else:
    print('Log directory not found. Run training first.')

## Ringkasan Side Quest TensorBoard

Pada notebook ini, TensorBoard digunakan untuk memantau proses pelatihan model secara visual. Callback `tf.keras.callbacks.TensorBoard` ditambahkan pada proses `model.fit()` agar metrik training dan validation tercatat otomatis.

Log yang dihasilkan disimpan dalam folder `meal_plan_artifacts_tensorboard/logs/fit` dan dapat dikompresi menjadi `tensorboard_logs.zip` untuk disertakan dalam repository akhir.

## 24. Catatan untuk Backend / Fullstack

File yang perlu diberikan ke backend jika model akan diintegrasikan:
- `meal_plan_model.keras`
- `feature_columns.json`
- `meal_plan_mapping.json`
- `scaler_info.json`

Input dari user harus diproses dengan alur:
1. Terima data asli user.
2. Normalisasi fitur numerik menggunakan `scaler_info.json`.
3. One-hot encode fitur kategorikal.
4. Susun kolom sesuai `feature_columns.json`.
5. Kirim ke model untuk prediksi.
6. Kembalikan output kelas dan probabilitas.